In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

from pynq import Overlay, allocate


# ============================================================
# Fixed point Q8.24
# ============================================================

Q_FRAC_BITS = 24
Q_SCALE = 1 << Q_FRAC_BITS


def float_to_q824(x):
    return np.int32(np.round(x * Q_SCALE))


def q824_to_float(x):
    return x.astype(np.float64) / Q_SCALE


# ============================================================
# State packing
#
# bits 31:0   = u
# bits 63:32  = v
# ============================================================

def pack_state(u, v):
    u_bits = int(np.uint32(float_to_q824(u)))
    v_bits = int(np.uint32(float_to_q824(v)))

    return np.uint64(
        u_bits | (v_bits << 32)
    )


def unpack_states(words):
    words = np.asarray(words, dtype=np.uint64)

    u_bits = (words & np.uint64(0xffffffff)).astype(np.uint32)
    v_bits = (words >> np.uint64(32)).astype(np.uint32)

    u_int = u_bits.view(np.int32)
    v_int = v_bits.view(np.int32)

    u = q824_to_float(u_int)
    v = q824_to_float(v_int)

    return u, v


# ============================================================
# Edge packing
#
# C++ expects:
#
# bits 31:0 = from
# bit  32   = last
#
# Edges MUST be grouped by destination.
#
# Since destination is implicit, for each destination n:
#
#   left_neighbor  -> n   last=0
#   right_neighbor -> n   last=1
#
# ============================================================

def pack_edge(source, last):
    word = int(source) & 0xffffffff

    if last:
        word |= (1 << 32)

    return np.uint64(word)


def build_bidirectional_ring(N):
    """
    For every neuron n:

        n-1 -> n
        n+1 -> n

    Therefore every neuron has:
        in-degree  = 2
        out-degree = 2

    Edge ordering is by destination.
    """

    edges = np.zeros(2 * N, dtype=np.uint64)
    out_degrees = np.full(N, 2, dtype=np.uint32)

    e = 0

    for n in range(N):
        left = (n - 1) % N
        right = (n + 1) % N

        edges[e] = pack_edge(left, False)
        e += 1

        edges[e] = pack_edge(right, True)
        e += 1

    return edges, out_degrees


# ============================================================
# Test
# ============================================================

def test_ring(
    bitfile,
    N=1024,
    iterations=20000,
    dt=1e-3,
    J=0.5,
    a=1.3,
    epsilon=0.1,
    impulse_width=5,
    impulse_u=1.5,
):

    # --------------------------------------------------------
    # Model parameters
    # --------------------------------------------------------

    inv_e = 1.0 / epsilon

    # Disable noise for first validation test
    sigma_sqrt_dt = 0.0

    seed = 12345678


    # --------------------------------------------------------
    # Resting fixed point
    # --------------------------------------------------------

    u_rest = -a
    v_rest = u_rest - u_rest**3 / 3.0

    print("Resting state:")
    print("u* =", u_rest)
    print("v* =", v_rest)


    # --------------------------------------------------------
    # Load overlay
    # --------------------------------------------------------

    ol = Overlay(bitfile)

    # Adjust this name if your IP has another name
    accel = ol.net_accel_0

    print(accel.register_map)


    # --------------------------------------------------------
    # Build graph
    # --------------------------------------------------------

    edges_host, degrees_host = build_bidirectional_ring(N)

    E = len(edges_host)

    print("N =", N)
    print("E =", E)


    # --------------------------------------------------------
    # Allocate physically contiguous buffers
    # --------------------------------------------------------

    state_in = allocate(
        shape=(N,),
        dtype=np.uint64
    )

    state_out = allocate(
        shape=(N,),
        dtype=np.uint64
    )

    edge_list = allocate(
        shape=(E,),
        dtype=np.uint64
    )

    out_degrees = allocate(
        shape=(N,),
        dtype=np.uint32
    )


    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    for n in range(N):
        state_in[n] = pack_state(u_rest, v_rest)


    # Excite a small patch around neuron 0.
    #
    # Using a patch is more robust than exciting exactly one
    # neuron for the first validation test.

    for k in range(-impulse_width, impulse_width + 1):
        n = k % N
        state_in[n] = pack_state(impulse_u, v_rest)


    # --------------------------------------------------------
    # Copy topology
    # --------------------------------------------------------

    edge_list[:] = edges_host
    out_degrees[:] = degrees_host


    # --------------------------------------------------------
    # Flush CPU caches
    # --------------------------------------------------------

    state_in.flush()
    edge_list.flush()
    out_degrees.flush()


    # --------------------------------------------------------
    # Configure accelerator
    # --------------------------------------------------------

    accel.register_map.state_in_1.state_in = \
        state_in.physical_address

    accel.register_map.edge_list_1.edge_list = \
        edge_list.physical_address

    accel.register_map.out_degrees_in_1.out_degrees = \
        out_degrees.physical_address

    accel.register_map.state_out_1.state_out = \
        state_out.physical_address


    accel.register_map.neuron_count = N
    accel.register_map.edge_count = E
    accel.register_map.iteration_count = iterations

    accel.register_map.dt = int(float_to_q824(dt))
    accel.register_map.J = int(float_to_q824(J))
    accel.register_map.a = int(float_to_q824(a))
    accel.register_map.inv_e = int(float_to_q824(inv_e))

    accel.register_map.sigma_sqrt_dt = \
        int(float_to_q824(sigma_sqrt_dt))

    accel.register_map.seed = seed
    accel.register_map.reseed = 1


    # --------------------------------------------------------
    # Run once
    # --------------------------------------------------------

    t0 = time.perf_counter()

    accel.register_map.CTRL.AP_START = 1

    while not accel.register_map.CTRL.AP_DONE:
        pass

    elapsed = time.perf_counter() - t0

    print(f"FPGA runtime: {elapsed:.6f} s")


    # --------------------------------------------------------
    # Read result
    # --------------------------------------------------------

    state_out.invalidate()

    u_final, v_final = unpack_states(
        state_out.copy()
    )


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    x = np.arange(N)

    plt.figure(figsize=(12, 4))
    plt.plot(x, u_final)
    plt.xlabel("Neuron index")
    plt.ylabel("u")
    plt.title(
        f"Ring FHN network after t={iterations * dt:.3f}"
    )
    plt.grid()
    plt.show()


    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    state_in.close()
    state_out.close()
    edge_list.close()
    out_degrees.close()

    return u_final, v_final

In [ ]:
u, v = test_ring(
    "flows_24k.bit",
    N=2024,
    iterations=500,
    dt=1e-3,
    J=0.5,
    impulse_width=5,
    impulse_u=1.5,
)

Resting state:
u* = -1.3
v* = -0.5676666666666667
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  state_in_1 = Register(state_in=write-only),
  state_in_2 = Register(state_in=write-only),
  edge_list_1 = Register(edge_list=write-only),
  edge_list_2 = Register(edge_list=write-only),
  out_degrees_in_1 = Register(out_degrees_in=write-only),
  out_degrees_in_2 = Register(out_degrees_in=write-only),
  state_out_1 = Register(state_out=write-only),
  state_out_2 = Register(state_out=write-only),
  neuron_count = Register(neuron_count=write-only),
  edge_count = Register(edge_count=write-only),
  iteration_count = Register(iteration_count=write-only),
  dt = Register(dt=write-only),
  J = Register(J=write-only),
  a =